# Session 16 Tasks — Python Solutions

This notebook follows the two tasks in `Session 16 Tasks.pdf`.

1. **Carrefour product scraper:** collect product name, price, product URL, and (when available) image, discount price, and category. Save results as CSV and JSON.
2. **Facebook Selenium automation:** open Facebook, let the account owner sign in manually, prepare a text post, and require a final confirmation before publishing.

**Important:** Carrefour pages may render products with JavaScript or block automated requests, and site markup changes. Use a category page you can access and are permitted to scrape, and follow the website's terms. Facebook selectors can also change; the script pauses for manual login and requires confirmation before it clicks Publish/Post. Do not store passwords in the code.


## 0. Install and import libraries

In [ ]:
# If needed, run this in Jupyter:
# %pip install -r requirements.txt

import json
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}


## Task 1 — Scrape Carrefour product information

Paste the URL of a Carrefour category page you can access into `CATEGORY_URL`. The scraper tries common product-card patterns and metadata. Since Carrefour page layouts differ by country and may load products dynamically, inspect the page HTML and adjust the CSS selectors if the first run returns no products.

In [ ]:
# REQUIRED: replace this with the actual Carrefour category page URL.
CATEGORY_URL = "PASTE_CARREFOUR_CATEGORY_URL_HERE"
OUTPUT_DIR = Path("task1_output")
OUTPUT_DIR.mkdir(exist_ok=True)

def clean_text(value):
    return re.sub(r"\\s+", " ", value or "").strip()

def get_first_text(element, selectors):
    for selector in selectors:
        found = element.select_one(selector)
        if found:
            text = clean_text(found.get_text(" ", strip=True))
            if text:
                return text
    return ""

def get_first_attr(element, selectors, attr):
    for selector in selectors:
        found = element.select_one(selector)
        if found and found.get(attr):
            return found.get(attr).strip()
    return ""

def parse_price(text):
    """Return a cleaned price string; keep currency/format rather than guessing a number."""
    return clean_text(text)

def scrape_carrefour_category(url):
    if not url.startswith(("http://", "https://")) or "PASTE_CARREFOUR" in url:
        raise ValueError("Set CATEGORY_URL to a real Carrefour category page URL first.")

    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Common selectors used by retail product cards. Change these if the site markup differs.
    card_selectors = [
        "[data-testid*='product-card']",
        "[data-product-id]",
        ".product-card",
        ".product",
        "article"
    ]
    cards = []
    for selector in card_selectors:
        found = soup.select(selector)
        if len(found) > len(cards):
            cards = found
        if len(cards) >= 3:
            break

    results = []
    seen_urls = set()
    for card in cards:
        name = get_first_text(card, [
            "[data-testid*='product-name']", ".product-name", ".product-title",
            "h2", "h3", "[class*='title']", "[class*='name']"
        ])
        product_href = get_first_attr(card, [
            "a[href*='/p/']", "a[href*='/product']", "a[href]"
        ], "href")
        product_url = urljoin(url, product_href) if product_href else ""
        price = get_first_text(card, [
            "[data-testid*='price']", "[class*='price']", ".price", "[itemprop='price']"
        ])
        image_src = get_first_attr(card, [
            "img[src]", "img[data-src]", "img[srcset]"
        ], "src") or get_first_attr(card, ["img[data-src]"], "data-src")
        if image_src and "," in image_src and " " in image_src:
            image_src = image_src.split(",")[0].strip().split(" ")[0]
        image_url = urljoin(url, image_src) if image_src else ""
        category = get_first_text(card, ["[class*='category']", "[itemprop='category']"])
        discount_price = get_first_text(card, [
            "[class*='discount']", "[class*='promo-price']", "[class*='special-price']"
        ])

        # Skip cards that do not appear to contain product data.
        if not name and not product_url:
            continue
        if product_url and product_url in seen_urls:
            continue
        if product_url:
            seen_urls.add(product_url)

        results.append({
            "product_name": name or "Not found",
            "product_price": parse_price(price) or "Not found",
            "product_url": product_url,
            "product_image": image_url,
            "discount_price": discount_price or "Not found",
            "product_category": category or "Not found"
        })

    # Fallback: inspect JSON-LD Product objects embedded in the HTML.
    if not results:
        def walk_json(obj):
            if isinstance(obj, dict):
                typ = obj.get("@type", [])
                if isinstance(typ, str):
                    typ = [typ]
                if "Product" in typ:
                    offer = obj.get("offers", {})
                    if isinstance(offer, list):
                        offer = offer[0] if offer else {}
                    image = obj.get("image", "")
                    if isinstance(image, list):
                        image = image[0] if image else ""
                    results.append({
                        "product_name": obj.get("name", "Not found"),
                        "product_price": str(offer.get("price", "Not found")) if isinstance(offer, dict) else "Not found",
                        "product_url": urljoin(url, obj.get("url", "")),
                        "product_image": urljoin(url, image) if image else "",
                        "discount_price": "Not found",
                        "product_category": obj.get("category", "Not found")
                    })
                for value in obj.values():
                    walk_json(value)
            elif isinstance(obj, list):
                for value in obj:
                    walk_json(value)

        for script in soup.select('script[type="application/ld+json"]'):
            try:
                walk_json(json.loads(script.string or script.get_text()))
            except (json.JSONDecodeError, TypeError):
                pass

    return results

products = scrape_carrefour_category(CATEGORY_URL)
products_df = pd.DataFrame(products, columns=[
    "product_name", "product_price", "product_url",
    "product_image", "discount_price", "product_category"
])
display(products_df.head(20))
print(f"Products collected: {len(products_df)}")
if products_df.empty:
    print("No products detected. This may be a JavaScript-rendered page or the selectors may need updating.")


In [ ]:
# Save both required structured formats: CSV and JSON.
csv_path = OUTPUT_DIR / "carrefour_products.csv"
json_path = OUTPUT_DIR / "carrefour_products.json"

products_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(products, f, ensure_ascii=False, indent=2)

print("Saved:", csv_path)
print("Saved:", json_path)


## Task 2 — Post a Facebook message using Selenium

This automation opens Facebook in a Chrome browser. You sign in yourself; credentials are never placed in the notebook. Because Facebook's interface changes frequently, this code uses accessible labels and fallback selectors. It prepares the post and **does not publish unless you type `POST` in the terminal prompt**. Use only an account you own or are authorized to operate.

In [ ]:
# Install ChromeDriver / Selenium as needed. Selenium Manager in modern Selenium
# can usually locate or download a compatible driver automatically.
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException

POST_TEXT = "Hello! This is a test post created for my Selenium practice task."

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
# Keep the browser visible so you can sign in and review the post.
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 30)

driver.get("https://www.facebook.com/")
input("Log in to Facebook in the opened browser (if needed), then return here and press Enter...")

print("Current page:", driver.current_url)
print("If the home feed is open, the script will try to open the Create post composer.")


In [ ]:
# Try common ways to open the post composer.
composer_opened = False
open_post_selectors = [
    (By.XPATH, "//span[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'what') and contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'mind')]"),
    (By.XPATH, "//div[@role='button'][.//*[contains(., 'Create post')]]"),
    (By.XPATH, "//span[contains(., 'Create post')]"),
]

for by, selector in open_post_selectors:
    try:
        element = wait.until(EC.element_to_be_clickable((by, selector)))
        element.click()
        composer_opened = True
        break
    except TimeoutException:
        continue

if not composer_opened:
    print("Could not automatically open the composer. Open the Facebook post composer manually in the browser.")
    input("When the Create post window is open, press Enter here...")

# Locate a text editor in the composer.
editor = None
editor_selectors = [
    (By.XPATH, "//div[@role='dialog']//div[@role='textbox' and @contenteditable='true']"),
    (By.XPATH, "//div[@role='textbox' and @contenteditable='true']"),
    (By.CSS_SELECTOR, "textarea")
]
for by, selector in editor_selectors:
    try:
        editor = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((by, selector)))
        break
    except TimeoutException:
        continue

if editor is None:
    raise RuntimeError("Could not find the post text box. Open the composer and update the selectors for Facebook's current interface.")

editor.click()
editor.send_keys(POST_TEXT)
print("Post text entered. Review it in the browser before publishing.")


In [ ]:
# Explicit safety checkpoint: publishing is an external action.
confirmation = input("Type POST to publish this message, or anything else to cancel: ").strip()

if confirmation == "POST":
    publish_button = None
    button_selectors = [
        (By.XPATH, "//div[@role='dialog']//div[@role='button'][.//*[normalize-space()='Post']]"),
        (By.XPATH, "//div[@role='dialog']//span[normalize-space()='Post']/ancestor::*[@role='button'][1]"),
        (By.XPATH, "//div[@role='dialog']//button[normalize-space()='Post']"),
    ]
    for by, selector in button_selectors:
        try:
            candidate = WebDriverWait(driver, 8).until(EC.element_to_be_clickable((by, selector)))
            publish_button = candidate
            break
        except TimeoutException:
            continue

    if publish_button is None:
        print("Post button not found. Review the browser and click Post manually if you still want to publish.")
    else:
        publish_button.click()
        print("Clicked Post. Check the Facebook browser to verify whether it was published.")
else:
    print("Cancelled. The script did not click the Post button.")

# When finished, close the browser with:
# driver.quit()


## Submission checklist

The task PDF requests source code, `requirements.txt`, screenshots showing output, and a short README for each task.

- Run Task 1 with a real Carrefour category URL and inspect the resulting CSV/JSON.
- Run Task 2 with your own/authorized Facebook account. Keep the browser visible and review the message before typing `POST`.
- Take screenshots of actual outputs after running the notebook; this package does not contain fabricated screenshots.
- If a site's page markup differs, update the selectors based on the page you are authorized to view.
